# 第1课：Iowa玉米产量趋势模型

这一课只做一件事：使用1996–2025年的Iowa玉米产量，估计长期趋势，并计算2026年**尚未加入天气影响**时的趋势产量。

请从上到下逐个运行代码单元格，不要使用 `Run All`。

## 0. 本课模型

我们要估计：

$$TrendYield_t = Intercept + Slope \times TrendIndex_t$$

其中：

$$TrendIndex_t = Year_t - 1996$$

因此1996年的Trend Index是0，2026年是30。

## 1. 导入Python工具

- `pandas`：整理表格数据
- `numpy`：完成数学计算
- `Path` 和 `json`：保存本课结果

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd

START_YEAR = 1996
END_YEAR = 2025
FORECAST_YEAR = 2026

print('工具导入成功。')

## 2. 建立历史产量表

为了让这个Notebook下载后可以直接运行，本教学版本内置了已经核对过的USDA NASS Iowa年度产量。最终项目仍保留完整CSV和逐年来源URL。

In [ ]:
years = list(range(1996, 2026))
yield_bu_per_acre = [
    138.0, 138.0, 145.0, 149.0, 144.0, 146.0, 163.0, 157.0,
    181.0, 173.0, 166.0, 171.0, 171.0, 181.0, 165.0, 172.0,
    137.0, 164.0, 178.0, 192.0, 203.0, 202.0, 196.0, 198.0,
    177.0, 204.0, 200.0, 201.0, 211.0, 210.0
]

data = pd.DataFrame({
    'year': years,
    'yield_bu_per_acre': yield_bu_per_acre,
})

data.head()

## 3. 检查数据

正式计算前先确认：

1. 是否正好有30年；
2. 是否覆盖1996–2025；
3. 是否有缺失值；
4. 是否有重复年份。

In [ ]:
assert len(data) == 30, '历史数据应该有30行。'
assert data['year'].tolist() == list(range(1996, 2026)), '年份必须连续覆盖1996–2025。'
assert not data.isna().any().any(), '数据中不能有缺失值。'
assert not data['year'].duplicated().any(), '年份不能重复。'

print('数据检查通过：30个年份、无缺失、无重复。')

## 4. 计算Trend Index

我们不用1996、1997这样的巨大年份直接做回归，而是把1996设为0。这样截距就可以解释为1996年的趋势产量。

In [ ]:
data['trend_index'] = data['year'] - START_YEAR
data[['year', 'yield_bu_per_acre', 'trend_index']].head()

## 5. 手动计算OLS斜率和截距

斜率公式：

$$Slope = \frac{\sum(x_i-\bar{x})(y_i-\bar{y})}{\sum(x_i-\bar{x})^2}$$

截距公式：

$$Intercept = \bar{y} - Slope \times \bar{x}$$

In [ ]:
x = data['trend_index'].to_numpy(dtype=float)
y = data['yield_bu_per_acre'].to_numpy(dtype=float)

x_mean = x.mean()
y_mean = y.mean()

slope_numerator = ((x - x_mean) * (y - y_mean)).sum()
slope_denominator = ((x - x_mean) ** 2).sum()

slope = slope_numerator / slope_denominator
intercept = y_mean - slope * x_mean

print(f'Intercept = {intercept:.4f}')
print(f'Slope = {slope:.4f} bu/acre per year')
print(f'Equation: Trend Yield = {intercept:.4f} + {slope:.4f} × Trend Index')

### 怎样解释Slope？

如果Slope约为2.31，表示在这条长期趋势线上，Iowa玉米单产平均每年增加约2.31 bushels/acre。它不代表每一年都一定增加2.31。

## 6. 计算每年的趋势产量和Residual

$$Residual = ActualYield - TrendYield$$

Residual为正表示实际产量高于趋势；为负表示实际产量低于趋势。

In [ ]:
data['trend_yield_bu_per_acre'] = intercept + slope * data['trend_index']
data['trend_residual_bu_per_acre'] = (
    data['yield_bu_per_acre'] - data['trend_yield_bu_per_acre']
)

data.tail().round(2)

## 7. 检查模型

- `R-squared`：趋势解释了多少历史产量波动；
- `RMSE`：趋势预测与实际产量的典型误差；
- 平均Residual：包含截距的OLS模型中应该非常接近0。

In [ ]:
residuals = data['trend_residual_bu_per_acre'].to_numpy()
sse = (residuals ** 2).sum()
sst = ((y - y_mean) ** 2).sum()
r_squared = 1 - sse / sst
rmse = np.sqrt(np.mean(residuals ** 2))
residual_mean = residuals.mean()

numpy_slope, numpy_intercept = np.polyfit(x, y, 1)
assert np.allclose([slope, intercept], [numpy_slope, numpy_intercept], atol=1e-10)

print(f'R-squared = {r_squared:.4f}')
print(f'RMSE = {rmse:.2f} bu/acre')
print(f'Mean residual = {residual_mean:.10f} bu/acre')
print('手动OLS与NumPy交叉检查一致。')

## 8. 预测2026年的趋势产量

2026年的Trend Index为：

$$2026-1996=30$$

然后把30放入趋势公式。这仍然不是最终模拟产量，因为天气还没有加入。

In [ ]:
forecast_trend_index = FORECAST_YEAR - START_YEAR
forecast_2026_trend_yield = intercept + slope * forecast_trend_index

print(f'2026 Trend Index = {forecast_trend_index}')
print(f'2026 Trend Yield = {forecast_2026_trend_yield:.2f} bu/acre')
print('注意：这只是长期趋势，尚未加入PDSI天气影响。')

## 9. 保存本课结果

运行下面单元格后，Notebook所在位置会出现 `lesson_output` 文件夹。

In [ ]:
output_dir = Path.cwd() / 'lesson_output'
output_dir.mkdir(parents=True, exist_ok=True)

data.to_csv(output_dir / 'yield_trend_1996_2025.csv', index=False)

summary = {
    'intercept_bu_per_acre': float(intercept),
    'annual_trend_slope_bu_per_acre': float(slope),
    'r_squared': float(r_squared),
    'rmse_bu_per_acre': float(rmse),
    'forecast_2026_trend_yield_bu_per_acre': float(forecast_2026_trend_yield),
    'important_limit': 'Weather has not been added yet.',
}

with (output_dir / 'yield_trend_summary.json').open('w', encoding='utf-8') as file:
    json.dump(summary, file, indent=2)

print('保存完成：')
print(output_dir / 'yield_trend_1996_2025.csv')
print(output_dir / 'yield_trend_summary.json')

## 本课结论

运行正确时，你应该得到：

- Intercept约为 **140.9269**；
- Slope约为 **2.3108 bu/acre/year**；
- 2026 trend yield约为 **210.25 bu/acre**。

到这里停止。下一课才会把July PDSI加入产量模型。